<a href="https://colab.research.google.com/github/JosueSandovalZT/Practicas_hechas_en_clase_U3/blob/main/Matematicas_Avanzadas_U3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import scipy.signal as signal
from scipy.integrate import solve_ivp

# Fowardpropagation

In [ ]:
import numpy as np

np.random.seed(42)

n_in, n_hidden, n_out = 2, 3, 1

# Inicialización Xavier
W1 = np.random.randn(n_in, n_hidden) * np.sqrt(2.0 / n_in)
b1 = np.zeros((1, n_hidden))
W2 = np.random.randn(n_hidden, n_out) * np.sqrt(2.0 / n_hidden)
b2 = np.zeros((1, n_out))

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

# Forward Propagation
X = np.array([[0.5, -0.2]])
z1 = np.dot(X, W1) + b1
a1 = sigmoid(z1)
z2 = np.dot(a1, W2) + b2
y_hat = sigmoid(z2)

print("Activación Oculta (a1):", a1)
print("Predicción Final (y_hat):", y_hat)

Activación Oculta (a1): [[0.48594148 0.49442486 0.59162127]]
Predicción Final (y_hat): [[0.67031527]]


In [ ]:
def sigmoid_deriv(a):
    return a * (1.0 - a)

y = np.array([[1.0]])
learning_rate = 0.5

# 1. Error y gradiente en capa de salida
error = y_hat - y
delta2 = error * sigmoid_deriv(y_hat)
dw2 = np.dot(a1.T, delta2)
db2 = np.sum(delta2, axis=0, keepdims=True)

# 2. Propagación del error a la capa oculta
delta1 = np.dot(delta2, W2.T) * sigmoid_deriv(a1)
dw1 = np.dot(X.T, delta1)
db1 = np.sum(delta1, axis=0, keepdims=True)

# 3. Actualización de pesos
W2 -= learning_rate * dw2
b2 -= learning_rate * db2
W1 -= learning_rate * dw1
b1 -= learning_rate * db1

print("Gradiente dw2:\n", dw2)
print("Gradiente dw1:\n", dw1)

Gradiente dw2:
 [[-0.03540469]
 [-0.03602277]
 [-0.0431043 ]]
Gradiente dw1:
 [[-0.01173379 -0.00570596  0.0033738 ]
 [ 0.00469352  0.00228238 -0.00134952]]


# Sensibilidad

In [ ]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def sigmoid_derivative(z):
    s = sigmoid(z)
    return s * (1.0 - s)

def neuron(x, w, b, activation='sigmoid'):
    z = np.dot(w, x) + b
    if activation == 'sigmoid':
        y = sigmoid(z)
        sens = sigmoid_derivative(z)
        return z, y, sens

# Ejemplo de evaluación
x = np.array([1.5, -0.5])
w = np.array([0.8, 1.2])
b = -0.2

z, y, sens = neuron(x, w, b)
print(f"z={z:.3f}, y={y:.3f}, df/dz={sens:.3f}")

z=0.400, y=0.599, df/dz=0.240


# Primer Orden

In [ ]:
# Taller 1: Sistema de Primer Orden
import numpy as np
import scipy.signal as signal

# Definición del sistema: x' + 2x = 4u
# H(s) = 4 / (s + 2)
num = [4]
den = [1, 2]
system = signal.TransferFunction(num, den)

# Respuesta al escalón unitario
t, y = signal.step(system)

# Parámetros característicos
pole = -den[1] / den[0]          # Polo p = -2
tau = -1 / pole                  # Constante de tiempo tau = 0.5 s
y_final = num[0] / den[1]        # Valor estacionario = 2.0

print(f"Polo: {pole}, Tau: {tau}s, Final: {y_final}")

Polo: -2.0, Tau: 0.5s, Final: 2.0


# Segundo Orden

In [ ]:
# Taller 2: Sistema de Segundo Orden
# x'' + 3x' + 9x = 9u
# H(s) = 9 / (s^2 + 3s + 9)
import numpy as np
import scipy.signal as signal

num = [9]
den = [1, 3, 9]

wn = np.sqrt(den[2])
zeta = den[1] / (2 * wn)
poles = np.roots(den)

Mp = np.exp(-np.pi * zeta / np.sqrt(1 - zeta**2)) * 100

print(f"wn: {wn:.2f} rad/s, zeta: {zeta:.2f}")
print(f"Polos: {poles}")
print(f"Sobrepaso máximo Mp: {Mp:.2f}%")

wn: 3.00 rad/s, zeta: 0.50
Polos: [-1.5+2.59807621j -1.5-2.59807621j]
Sobrepaso máximo Mp: 16.30%


# Sistema Inestable

In [ ]:
# Análisis de Estabilidad - Sistema Inestable
import numpy as np
from scipy.integrate import solve_ivp

# Matriz A con un autovalor Re(lambda) > 0
A_in = np.array([[1.5, 2.0],
                 [-0.5, -1.0]])

eigvals_in, _ = np.linalg.eig(A_in)
print('Autovalores:', eigvals_in)
# Out: [1.0, -0.5]  (1.0 > 0 -> INESTABLE)

# Simular respuesta x(t)
def sys_un(t, x):
    return A_in @ x

sol_un = solve_ivp(sys_un, [0, 5], [0.1, 0.1])
print('Estado Final x(5):', sol_un.y[:, -1])

Autovalores: [ 1.  -0.5]
Estado Final x(5): [39.56138141 -9.88008338]


# Sistema Estable

In [ ]:
# Análisis de Estabilidad - Sistema Estable
import numpy as np
from scipy.integrate import solve_ivp

# Matriz A con Re(lambda) < 0
A = np.array([[-2.0,  1.0],
              [-1.0, -3.0]])

# Calcular autovalores
eigvals, _ = np.linalg.eig(A)
print('Autovalores:', eigvals)
# Out: [-1.618, -3.382] (Ambos < 0 -> ESTABLE)

# Simular respuesta x(t)
def system(t, x):
    return A @ x

sol = solve_ivp(system, [0, 5], [2.0, -1.0])
print('Estado Final x(5):', sol.y[:, -1])

Autovalores: [-2.5+0.8660254j -2.5-0.8660254j]
Estado Final x(5): [-2.82166076e-06  7.21983870e-06]
